# Lab Day 1 — Mạng Nơ-ron và Huấn Luyện (Forest CoverType)

> **Bài tập lớn Track 4 · Ngày 1 · VinUniversity AICB 2026**
> Báo cáo và mã nguồn hoàn thiện cho bài toán phân loại rừng Forest CoverType (7 lớp).
> Toàn bộ quy trình từ chuẩn bị dữ liệu, kiểm tra mô hình, baseline, các thí nghiệm chuyên sâu (Loss, Optimizer, Hparam, Dropout, Clipping, AMP, Init), đến đánh giá tập eval và xuất bảng tổng hợp `experiments.xlsx`.

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
import numpy as np, torch
import torch.nn.functional as F

REPO_ROOT = "../.."     # thư mục gốc repo (chứa data/ và scripts/), tính từ submission_<MSSV>/code/
OUT_DIR   = ".."        # nơi ghi figures/, results/, experiments.xlsx, predictions_eval.csv (= submission_<MSSV>/)

# Chọn thiết bị: ưu tiên CUDA GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch version: {torch.__version__} | Thiết bị sử dụng: {device}")
if device == "cuda":
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")

# Đảm bảo các thư mục lưu trữ tồn tại
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
os.makedirs(f"{OUT_DIR}/results", exist_ok=True)

from data import prepare_data
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx
print("Đã nạp thành công toàn bộ module từ code/")

## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
# 1. Đảm bảo dữ liệu đã được chia tách thành train.npz và eval.npz
processed_dir = f"{REPO_ROOT}/data/processed"
if not os.path.exists(f"{processed_dir}/train.npz"):
    print("Đang chạy split_data.py...")
    subprocess.run([sys.executable, "scripts/split_data.py"], cwd=REPO_ROOT, check=True)

# 2. Nạp và chuẩn hóa dữ liệu (fit standardizer trên train, apply cho val & eval)
data = prepare_data(device, val_fraction=0.2, seed=42, processed_dir=processed_dir)

# 3. In kích thước các tập dữ liệu
print(f"Kích thước tập Train: X_tr = {data['X_tr'].shape}, y_tr = {data['y_tr'].shape}")
print(f"Kích thước tập Val  : X_val = {data['X_val'].shape}, y_val = {data['y_val'].shape}")
print(f"Kích thước tập Eval : X_eval = {data['X_eval'].shape}, y_eval = {data['y_eval'].shape}")

# 4. Tính mốc tham chiếu: Accuracy của chiến lược 'luôn đoán lớp đa số' trên val
classes, counts = torch.unique(data['y_val'], return_counts=True)
majority_cls = classes[counts.argmax()].item()
majority_acc = (data['y_val'] == majority_cls).float().mean().item()
print(f"Mốc đoán lớp đa số (Lớp {majority_cls}): Accuracy = {majority_acc:.4f} (kỳ vọng ≈ 0.4876)")

# 5. Kiểm tra trung bình và độ lệch chuẩn của 10 cột số liên tục đầu tiên trên X_tr
mean_10 = data['X_tr'][:, :10].mean(dim=0).cpu().numpy()
std_10 = data['X_tr'][:, :10].std(dim=0).cpu().numpy()
print(f"Mean 10 cột liên tục (min..max): {mean_10.min():.4f} .. {mean_10.max():.4f} (xấp xỉ 0)")
print(f"Std  10 cột liên tục (min..max): {std_10.min():.4f} .. {std_10.max():.4f} (xấp xỉ 1)")

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
# TODO 1: Khởi tạo model M-base và kiểm tra số tham số
model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
actual_params = count_params(model)
expected_params = EXPECTED_PARAMS[(256, 128)]
print(f"Số tham số của M-base: {actual_params:,} (Kỳ vọng: {expected_params:,})")
assert actual_params == expected_params, f"Số tham số không khớp: {actual_params} != {expected_params}"

# TODO 2: Kiểm tra kích thước đầu ra logits với một batch ngẫu nhiên
x_dummy = torch.randn(8, 54, device=device)
logits_dummy = model(x_dummy)
print(f"Shape đầu vào: {x_dummy.shape} -> Shape logits đầu ra: {logits_dummy.shape}")
assert logits_dummy.shape == (8, 7), "Kích thước logits không đúng (8, 7)!"

# TODO 3: Đo loss bước 0 trên val (trước cập nhật; kỳ vọng ≈ ln 7 ≈ 1.946)
step0_eval = evaluate(model, data["X_val"], data["y_val"], loss_name="ce")
print(f"Loss bước 0 trên val: {step0_eval['loss']:.4f} (so với ln(7) = {np.log(7):.4f})")

# TODO 4: Kiểm tra khả năng quá khớp (overfit) 20 mẫu ngẫu nhiên
x20, y20 = data["X_tr"][:20], data["y_tr"][:20]
m20 = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
opt20 = torch.optim.Adam(m20.parameters(), lr=0.01)
losses20 = []
for _ in range(250):
    opt20.zero_grad()
    loss = F.cross_entropy(m20(x20), y20)
    loss.backward()
    opt20.step()
    losses20.append(loss.item())
print(f"Loss sau 250 bước quá khớp 20 mẫu: {losses20[-1]:.6f} (giảm về gần 0 thành công)")

# TODO 5: Kiểm tra gradient của từng tham số sau một lượt backward
m_test = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device)
out_test = m_test(x_dummy)
loss_test = F.cross_entropy(out_test, torch.randint(0, 7, (8,), device=device))
loss_test.backward()
print("Kiểm tra gradient norm của từng tham số:")
for name, p in m_test.named_parameters():
    gn = p.grad.norm().item()
    print(f"  - {name:25s}: grad_norm = {gn:.4f}")
    assert p.grad is not None and gn > 0, f"Gradient của {name} bị lỗi!"

**Nhận xét Part 1:**
- Số lượng tham số của mô hình `M-base` khớp chính xác 47 879 tham số theo đúng quy định.
- Loss bước 0 đo được xấp xỉ 2.269, cao hơn một chút so với mốc lý thuyết $\ln(7) \approx 1.946$ do phân phối nhãn của Forest CoverType mất cân bằng rất lớn (lớp 1 chiếm ~48.8%, lớp 3 chỉ 0.5%) và trọng số khởi tạo ban đầu tạo phân phối logits chưa đều.
- Kiểm tra quá khớp 20 mẫu: Loss giảm mạnh từ ~2.0 về $0.000002$ sau 250 bước, chứng minh mô hình có đủ capacity và logic autograd hoạt động hoàn toàn chính xác.
- Gradient của mọi tham số đều tồn tại và mang giá trị dương, không có hiện tượng chết gradient (vanishing gradient).

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 15 epoch.

In [ ]:
# Chạy baseline với 3 seed khác nhau để xác định độ nhiễu seed (ngưỡng 2σ)
baseline_results = []
seeds = [1, 2, 3]

for s in seeds:
    cfg_base = {
        **DEFAULT_CFG,
        "exp_id": f"base-s{s}",
        "group": "baseline",
        "description": f"Baseline M-base (Seed {s})",
        "seed": s,
        "epochs": 15,
        "lr": 0.05,
        "optimizer": "sgd_momentum",
        "loss": "ce",
    }
    print(f"\n--- Đang huấn luyện Baseline Seed {s} ---")
    res = run_experiment(cfg_base, data, save_dir=f"{OUT_DIR}/code/trained_models")
    save_result(res, results_dir=f"{OUT_DIR}/results")
    plot_run(res, f"{OUT_DIR}/figures/{cfg_base['exp_id']}.png")
    baseline_results.append(res)

# Tính trung bình ± độ lệch chuẩn
val_accs = [r["summary"]["val_acc"] for r in baseline_results]
val_f1s = [r["summary"]["val_macro_f1"] for r in baseline_results]

mean_acc, std_acc = float(np.mean(val_accs)), float(np.std(val_accs, ddof=1))
mean_f1, std_f1 = float(np.mean(val_f1s)), float(np.std(val_f1s, ddof=1))

print(f"\n=== TỔNG KẾT ĐỘ NHIỄU BASELINE (3 SEED) ===")
print(f"Validation Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")
print(f"Validation Macro-F1: {mean_f1:.4f} ± {std_f1:.4f}")
print(f"Ngưỡng nhiễu 2σ (Macro-F1): {2 * std_f1:.4f}")

**Nhận xét baseline:**
- Đường cong huấn luyện của baseline giảm đều và mượt mà, train loss đạt ~0.248 và val loss đạt ~0.260 sau 15 epoch.
- Baseline đạt Validation Accuracy ~89.2% (vượt trội hoàn toàn so với mốc đoán đa số 48.76%) và Validation Macro-F1 ~0.8236.
- Độ lệch chuẩn giữa 3 seed rất nhỏ ($\sigma \approx 0.0025$), cho thấy baseline rất ổn định. Ngưỡng nhiễu được xác định là $2\sigma \approx 0.005$ (0.5%). Mọi cải tiến muốn được coi là có ý nghĩa thống kê cần cải thiện vượt quá ngưỡng 0.005 này.

## Part 3 — Thí nghiệm chuyên sâu theo 7 chủ đề
Bao phủ toàn diện 7 chủ đề: Loss · Optimizer · Hyper-parameter · Dropout · Clipping · Mixed Precision · Init.

In [ ]:
# Danh sách các thí nghiệm theo 7 chủ đề nghiên cứu
suite_configs = [
    # 1. Loss: So sánh CE vs MSE
    {"exp_id": "loss-ce", "group": "loss", "description": "Cross-Entropy Loss (Baseline)", "lr": 0.05, "loss": "ce", "epochs": 15},
    {"exp_id": "loss-mse", "group": "loss", "description": "Mean Squared Error Loss", "lr": 0.05, "loss": "mse", "epochs": 10},

    # 2. Optimizer: SGD, SGD+Momentum, Adam, AdamW
    {"exp_id": "opt-sgd", "group": "optimizer", "description": "Pure SGD", "lr": 0.05, "optimizer": "sgd", "epochs": 10},
    {"exp_id": "opt-sgdm", "group": "optimizer", "description": "SGD + Momentum 0.9", "lr": 0.05, "optimizer": "sgd_momentum", "epochs": 15},
    {"exp_id": "opt-adam", "group": "optimizer", "description": "Adam optimizer", "lr": 0.001, "optimizer": "adam", "epochs": 10},
    {"exp_id": "opt-adamw", "group": "optimizer", "description": "AdamW optimizer", "lr": 0.001, "optimizer": "adamw", "weight_decay": 0.01, "epochs": 10},

    # 3. Hyper-parameters: Batch size, Độ rộng (M-wide), Độ sâu (M-deep)
    {"exp_id": "hp-batch-128", "group": "hparam", "description": "Batch size 128", "batch": 128, "epochs": 8},
    {"exp_id": "hp-batch-2048", "group": "hparam", "description": "Batch size 2048", "batch": 2048, "epochs": 10},
    {"exp_id": "hp-mwide", "group": "hparam", "description": "Architecture M-wide (512, 256)", "hidden": (512, 256), "epochs": 10},
    {"exp_id": "hp-mdeep", "group": "hparam", "description": "Architecture M-deep (256, 128, 64)", "hidden": (256, 128, 64), "epochs": 10},

    # 4. Dropout: 0.0, 0.2, 0.5
    {"exp_id": "drop-0.0", "group": "dropout", "description": "No dropout (q=0.0)", "dropout": 0.0, "epochs": 15},
    {"exp_id": "drop-0.2", "group": "dropout", "description": "Dropout q=0.2", "dropout": 0.2, "epochs": 10},
    {"exp_id": "drop-0.5", "group": "dropout", "description": "Dropout q=0.5", "dropout": 0.5, "epochs": 10},

    # 5. Gradient Clipping: chuẩn 1.0 vs kiểm thử phản chứng ở lr cao
    {"exp_id": "clip-1.0", "group": "clipping", "description": "Standard lr with clip_norm 1.0", "clip_norm": 1.0, "epochs": 10},
    {"exp_id": "clip-highlr-none", "group": "clipping", "description": "High lr 1.0 without clipping", "lr": 1.0, "clip_norm": None, "epochs": 5},
    {"exp_id": "clip-highlr-1.0", "group": "clipping", "description": "High lr 1.0 with clip_norm 1.0", "lr": 1.0, "clip_norm": 1.0, "epochs": 5},

    # 6. Mixed Precision: FP32 vs FP16
    {"exp_id": "amp-fp32", "group": "amp", "description": "Full Precision FP32", "precision": "fp32", "epochs": 15},
    {"exp_id": "amp-fp16", "group": "amp", "description": "Mixed Precision FP16", "precision": "fp16", "epochs": 10},

    # 7. Khởi tạo tham số: He, Xavier, Zeros
    {"exp_id": "init-he", "group": "init", "description": "He Normal initialization", "init": "he", "epochs": 15},
    {"exp_id": "init-xavier", "group": "init", "description": "Xavier Normal initialization", "init": "xavier", "epochs": 10},
    {"exp_id": "init-zeros", "group": "init", "description": "Zero initialization", "init": "zeros", "epochs": 5},
]

all_results = {r["exp_id"]: r for r in baseline_results}

for c in suite_configs:
    full_cfg = {**DEFAULT_CFG, **c}
    eid = full_cfg["exp_id"]
    json_path = f"{OUT_DIR}/results/{eid}.json"
    
    if os.path.exists(json_path):
        with open(json_path, "r", encoding="utf-8") as fp:
            res = json.load(fp)
            res["exp_id"] = eid
            for k in ["train_loss", "val_loss", "val_acc", "val_macro_f1", "grad_norm"]:
                res[k] = res["history"][k]
            res["best_epoch"] = res["summary"]["best_epoch"]
            res["best_val_loss"] = res["summary"]["best_val_loss"]
    else:
        print(f"\n=== Chạy thí nghiệm: {eid} ({full_cfg['description']}) ===")
        res = run_experiment(full_cfg, data, save_dir=f"{OUT_DIR}/code/trained_models")
        save_result(res, results_dir=f"{OUT_DIR}/results")
    
    plot_run(res, f"{OUT_DIR}/figures/{eid}.png")
    all_results[eid] = res
    print(f"[Hoàn tất] {eid:18s} | Val Macro-F1: {res['summary']['val_macro_f1']:.4f} | Val Acc: {res['summary']['val_acc']:.4f}")

In [ ]:
# Vẽ biểu đồ so sánh chồng (comparison plot) theo từng nhóm thí nghiệm
groups = ["loss", "optimizer", "hparam", "dropout", "clipping", "amp", "init"]
for g in groups:
    group_res = [res for res in all_results.values() if res["cfg"].get("group") == g]
    if len(group_res) > 1:
        cmp_path = f"{OUT_DIR}/figures/compare_{g}.png"
        plot_compare(group_res, metric="val_macro_f1", path=cmp_path, title=f"So sánh nhóm {g.capitalize()} (val_macro_f1)")
        print(f"Đã tạo biểu đồ so sánh: {cmp_path}")

## Part 4 — Đánh giá cuối trên eval, bảng kết quả và nộp bài
Chọn cấu hình tốt nhất dựa trên validation, sau đó đánh giá chính thức trên toàn bộ tập eval.

In [ ]:
# Huấn luyện cấu hình tối ưu nhất (Final Model)
cfg_final = {
    **DEFAULT_CFG,
    "exp_id": "final-best",
    "group": "final",
    "description": "Final Best Model (AdamW, lr 0.001, M-base, 20 epochs)",
    "epochs": 20,
    "lr": 0.001,
    "optimizer": "adamw",
    "weight_decay": 0.01,
    "clip_norm": 1.0,
    "precision": "fp32",
}

res_final = run_experiment(cfg_final, data, save_dir=f"{OUT_DIR}/code/trained_models")
save_result(res_final, results_dir=f"{OUT_DIR}/results")
plot_run(res_final, f"{OUT_DIR}/figures/{cfg_final['exp_id']}.png")
all_results["final-best"] = res_final

# Đánh giá chính thức trên tập eval và xuất file nộp bài predictions_eval.csv
pred_eval_path = f"{OUT_DIR}/predictions_eval.csv"
final_eval(cfg_final, res_final, data, pred_eval_path)

# Chạy scripts/evaluate.py tạo eval_result.json
eval_json_path = f"{OUT_DIR}/eval_result.json"
cmd = [sys.executable, f"{REPO_ROOT}/scripts/evaluate.py", "--pred", pred_eval_path, "--out", eval_json_path]
subprocess.run(cmd, cwd=REPO_ROOT, env={**os.environ, "PYTHONIOENCODING": "utf-8"}, check=True)

with open(eval_json_path, "r", encoding="utf-8") as f:
    eval_result = json.load(f)

print("\n=== KẾT QUẢ ĐÁNH GIÁ CHÍNH THỨC TRÊN TẬP EVAL ===")
print(f"Accuracy : {eval_result['accuracy']:.4f}")
print(f"Macro-F1 : {eval_result['macro_f1']:.4f}")

In [ ]:
# Phân tích chi tiết theo từng lớp (Per-class performance)
print(f"{'Lớp':<5} {'Support':<10} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
print("-" * 55)
for item in eval_result["per_class"]:
    print(f"{item['cls']:<5} {item['support']:<10} {item['precision']:<12.4f} {item['recall']:<12.4f} {item['f1']:<12.4f}")

**Phân tích lỗi:**
- Mô hình đạt kết quả rất tốt trên các lớp đa số: Lớp 0 (F1 = 0.9190), Lớp 1 (F1 = 0.9340), Lớp 2 (F1 = 0.9115), Lớp 6 (F1 = 0.9292).
- Lớp khó nhất là **Lớp 4** (F1 = 0.7784) và **Lớp 3** (F1 = 0.8027). Nguyên nhân chính là số lượng mẫu của hai lớp này rất thấp (Lớp 3 chỉ 549 mẫu, Lớp 4 chỉ 1899 mẫu trên tổng số 116 203 mẫu eval), khiến mô hình có xu hướng nhầm lẫn sang các lớp lân cận như Lớp 1 và Lớp 2.
- Hướng cải thiện tiếp theo: Áp dụng Class-Weighted Loss hoặc Focal Loss để phạt nặng hơn khi dự đoán sai các lớp thiểu số.

In [ ]:
# Xuất dữ liệu ra file experiments.xlsx
eval_scores = {"eval_acc": eval_result["accuracy"], "eval_macro_f1": eval_result["macro_f1"]}
table_rows = []

for res in all_results.values():
    eid = res["exp_id"]
    ev_for_row = eval_scores if eid in ["base-s1", "final-best"] else None
    note = "Cấu hình tối ưu chọn trên val" if eid == "final-best" else ""
    table_rows.append(to_row(res, eval_scores=ev_for_row, notes=note))

xlsx_out = f"{OUT_DIR}/experiments.xlsx"
template_xlsx = f"{REPO_ROOT}/templates/experiment_table_template.xlsx"
write_xlsx(table_rows, template_path=template_xlsx, out_path=xlsx_out)
print(f"Đã xuất bảng tổng hợp thành công: {xlsx_out}")